# CRISP-DM Phase 5 — Evaluation (metrics + sanity check)

Reports the performance of the final 3-class model and a one-shot sanity
check on real Apple-Watch recordings. Metrics are loaded from the committed
`models/saved/model_metrics.json` (produced by `scripts/train_model.py`), so the
notebook reflects exactly what training produced — no dataset needed for this part.

Key decisions (see docs/DECISIONS.md): classes §3, leave-one-set-out + single-subject
limitation §6.

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np

from ml4b.data.apple_watch_loader import predict_from_sensor_logger
from ml4b.utils.config import DATA_PROCESSED, DATA_RAW, MODELS_DIR
from ml4b.utils.metrics import load_model_metrics

m = load_model_metrics()
print("Evaluation :", m["evaluation"])
print(f"Macro F1   : {m['cv_macro_f1']:.4f}")
print(f"Accuracy   : {m['cv_accuracy']:.4f}")
print("Per-class F1:")
for c, f in m["cv_per_class_f1"].items():
    print(f"  {c:<18} {f:.4f}")

## 1. Confusion matrix (leave-one-set-out)

In [ ]:
labels = m["confusion_matrix_labels"]
cm = np.array(m["confusion_matrix"], dtype=float)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix (row-normalized)")
for r in range(len(labels)):
    for col in range(len(labels)):
        ax.text(
            col,
            r,
            f"{cm_norm[r, col]:.2f}",
            ha="center",
            va="center",
            color="white" if cm_norm[r, col] > 0.5 else "black",
        )
fig.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 2. Limitations
- **Single-subject training anchor.** The Kaggle dataset is one person on one Apple
  Watch, so true leave-one-*subject*-out is impossible. The macro F1 above measures
  generalisation to an unseen **set**, not a new **person**.
- **Real-world performance will be below these numbers** for a different user.
- **Augmentation (DECISIONS.md)** synthesises the missing subject diversity — a documented
  mitigation, not a replacement for real multi-subject data (which was unavailable).
- The methodology is sound: Apple-Watch training domain (DECISIONS.md), leakage-free
  evaluation (DECISIONS.md), invariant features (DECISIONS.md), an energy gate for rest (DECISIONS.md),
  and confidence-based abstention (DECISIONS.md).

## 3. Sanity check on real Apple-Watch samples (run once, as-is)
Runs the deployed pipeline on `data/raw/apple_watch/test_samples/`. These files were
**not** used for training or tuning and **nothing is changed** in response to them.
`push_up` is out of scope (absent from training, DECISIONS.md) and is reported, not scored.
Full write-up: `docs/project/apple_watch_validation_results.md`.

In [ ]:
model = joblib.load(MODELS_DIR / "best_model.joblib")
feature_names = (DATA_PROCESSED / "feature_names.txt").read_text().strip().split("\n")
known = set(m["confusion_matrix_labels"])
samples_dir = DATA_RAW / "apple_watch" / "test_samples"
samples = sorted(samples_dir.glob("*.csv"))
print("samples found:", [s.name for s in samples])

for s in samples:
    true = s.name.replace(".csv", "").split("_sample")[0].split("_session")[0]
    res = predict_from_sensor_logger(s, model, feature_names)
    counts = res["predicted_class"].value_counts().to_dict()
    ex = res[~res["predicted_class"].isin({"rest", "uncertain"})]
    top = ex["predicted_class"].mode().iloc[0] if not ex.empty else "none"
    conf = (
        float(res["confidence"].mean())
        if res["confidence"].notna().any()
        else float("nan")
    )
    scope = "" if true in known else "  (OUT OF SCOPE)"
    print(f"\n{s.name}{scope}")
    print(f"  detected rate : {res.attrs.get('detected_hz')} Hz | windows: {len(res)}")
    print(f"  distribution  : {counts}")
    print(f"  top exercise  : {top} | avg confidence: {conf:.3f}")
    if true in known:
        print(f"  verdict       : {'CORRECT' if top == true else 'MISCLASSIFIED'}")
    else:
        print("  verdict       : n/a (class not in the 3-class scope)")

## 4. Interpretation
- Both **bicep-curl** recordings are correctly identified as `bicep_curl` (the key
  cross-recording success — the previous MM-Fit model failed here).
- Window-level **bicep vs triceps** confusion persists (both are elbow movements), but
  the majority label is correct.
- **Rest** gating and **uncertain** abstention behave sensibly on real data.
- **push_up** is out of scope and maps to its nearest class (tricep extension) at low
  confidence — a faithful 'I don't have this class' signal.

## 5. Summary
Leave-one-set-out macro F1 ~ 0.78; real Apple-Watch curls recognised correctly;
limitations documented. Deployment is covered in **06_streamlit_demo** and the app.